# Day 13: Agentic RAG, Chatting with Your Own Documents

If you've worked on an AI project at a company, there's a good chance it started with this request: "Can we have a chatbot that answers questions from **our** documents?" Our HR policy, our product manual, our support tickets.

The model doesn't know any of that. It was never trained on your company's PDFs. And pasting every document into every prompt isn't possible, it's far too much text.

The solution is called **RAG**, Retrieval-Augmented Generation. The idea is like an open-book exam:

1. **Retrieve**: before answering, flip to the few pages of the textbook that are relevant to the question.
2. **Augment**: put those pages in front of the model, along with the question.
3. **Generate**: the model writes the answer using those pages.

Today we'll build that step by step, and then make it smarter: an agent that decides *when* to search, checks whether what it found is actually useful, and searches again with a better query if it isn't.

About cost: we'll use OpenAI's `text-embedding-3-small` for embeddings. For a handful of short documents like ours, it costs a tiny fraction of a rupee.

In [ ]:
%pip install -q -U langgraph langchain langchain-openai langchain-text-splitters python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. How does the search know what's "relevant"?

This is the part that feels like magic, so let me explain it simply.

Imagine a huge library where books aren't arranged alphabetically, but by **meaning**. Books about cooking sit together, books about cricket sit together, and a book about cooking for cricket teams sits somewhere in between. If you walk to the spot for "how do I get my money back?", you'll find the refund policy right there, even though it never uses the words "money back".

**Embeddings** do exactly this. They turn a piece of text into a list of numbers, a position in that "library of meaning". Texts with similar meaning get similar numbers. A **vector store** keeps all those positions and quickly finds the ones closest to your question.

The full pipeline looks like this:

```
your documents -> split into small chunks -> embed each chunk -> store them
                                                                   |
question -> embed the question -> find the closest chunks ---------+
                                          |
                                          v
                         chunks + question -> LLM -> answer
```

## 2. Building a small knowledge base

Our documents are the handbook of a made-up online school, "Chai and Code Academy". Later you can swap these for your own text, or load PDFs.

In [ ]:
from langchain_core.documents import Document

In [ ]:
handbook = {
    "refunds.md": (
        "Refund policy. Students can get a full refund within 7 days of purchase if they have "
        "watched less than 20% of the course. After 7 days no refunds are given, but the course "
        "can be transferred once to a friend. Refunds go back to the original payment method "
        "within 5 working days."
    ),
    "certificates.md": (
        "Certificates. A certificate is issued when a student completes all lessons and scores at "
        "least 60% in the final quiz. The quiz can be attempted 3 times. Certificates are emailed "
        "as a PDF within 48 hours and can be verified on our website using the certificate ID."
    ),
    "schedule.md": (
        "Live classes. Live doubt-clearing sessions happen every Saturday at 7 PM IST on Zoom. "
        "Recordings are uploaded within 24 hours. The LangGraph batch starts on the 1st of every "
        "month and runs for 16 days, one lesson per day."
    ),
    "support.md": (
        "Support. Email help@chaiandcode.example for account issues. The Telegram community is the "
        "fastest way to get coding help. Mentors reply within 12 hours on working days. Office "
        "hours are Monday to Friday, 10 AM to 6 PM IST."
    ),
}

In [ ]:
# Wrap each text in a Document, and remember which file it came from
docs = [Document(page_content=text, metadata={"source": name}) for name, text in handbook.items()]
print(len(docs), "documents")

Next we **split** the documents into small chunks. Why? If a whole 50-page manual were one chunk, every search would return the entire manual. Small chunks mean each search result is focused on one idea.

`chunk_overlap` repeats a little text between neighbouring chunks, so a sentence that falls across a boundary isn't cut in half and lost.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)

print(len(chunks), "chunks")
print("Example chunk:", chunks[0].page_content)
print("It came from:", chunks[0].metadata["source"])

Now we embed every chunk and put them in a vector store. `InMemoryVectorStore` needs no database, which is perfect for learning. In real projects you'd use something like Chroma, FAISS, pgvector or Pinecone. The code looks almost the same.

A **retriever** is the "search" button on top of the store. `k=2` means "give me the 2 closest chunks".

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

In [ ]:
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

### Predict before you run

We'll search for "Can I get my money back?". Which file do you expect the top result to come from? Note that the words "money back" don't appear anywhere in the handbook.

<details>
<summary>Click to see the answer</summary>

`refunds.md`. The embedding search matches **meaning**, not exact words, and "money back" means the same thing as "refund".

That's the big advantage over a normal keyword search (like Ctrl+F).
</details>

In [ ]:
for doc in retriever.invoke("Can I get my money back?"):
    print(f"[{doc.metadata['source']}] {doc.page_content[:90]}...")

### Your turn

Try a few questions of your own. Try one using completely different words from the handbook, like "When can I talk to a teacher live?"

In [ ]:
my_question = "When can I talk to a teacher live?"        # change this

for doc in retriever.invoke(my_question):
    print(f"[{doc.metadata['source']}] {doc.page_content[:90]}...")

## 3. A simple RAG graph

Two nodes: find the relevant chunks, then answer using them.

```
  START -> retrieve -> generate -> END
```

The prompt in `generate` matters a lot. We tell the model to use **only** the context, to say "I don't know" when the answer isn't there, and to mention which file the answer came from. Without those instructions, the model happily fills gaps with confident made-up answers.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class RAGState(TypedDict):
    question: str
    context: list[Document]
    answer: str

In [ ]:
def retrieve(state: RAGState) -> dict:
    return {"context": retriever.invoke(state["question"])}

In [ ]:
def generate(state: RAGState) -> dict:
    context_text = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in state["context"])

    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer isn't in the context, say \"I don't know\". "
        "Mention the source file in square brackets.\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {state['question']}"
    )
    return {"answer": llm.invoke(prompt).content}

In [ ]:
builder = StateGraph(RAGState)

In [ ]:
builder.add_sequence([retrieve, generate])

In [ ]:
builder.add_edge(START, "retrieve")
builder.add_edge("generate", END)

In [ ]:
rag_app = builder.compile()

### Predict before you run

The third question below is "Who won the 2011 Cricket World Cup?". The model definitely knows the answer from its training. Will it answer?

<details>
<summary>Click to see the answer</summary>

It should say "I don't know", because we told it to use only the context, and the handbook says nothing about cricket.

That might feel strange, but it's exactly what you want in a company chatbot. Users must be able to trust that the answer came from your documents, not from the model's general knowledge.
</details>

In [ ]:
questions = [
    "How many times can I attempt the final quiz?",
    "When are the live classes?",
    "Who won the 2011 Cricket World Cup?",
]

for question in questions:
    result = rag_app.invoke({"question": question})
    print("Q:", question)
    print("A:", result["answer"])
    print()

## 4. Agentic RAG: let the model decide when to search

The simple version above has a weakness: it **always** searches, even when the user just says "hi". And if a question needs two searches ("compare the refund rules with the certificate rules"), it can only do one.

The fix uses everything from Day 6. We wrap the retriever in a `@tool`, and let an agent decide when (and how many times) to call it.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
@tool
def search_handbook(query: str) -> str:
    """Search the Chai and Code Academy handbook: refunds, certificates, live classes and support."""
    found = retriever.invoke(query)
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in found)

In [ ]:
rag_tools = [search_handbook]
agent_llm = llm.bind_tools(rag_tools)

AGENT_PROMPT = SystemMessage(content=(
    "You are the Chai and Code Academy assistant. For questions about the academy, "
    "ALWAYS use search_handbook and mention the source file. For small talk, just reply normally."
))

In [ ]:
def agent(state: MessagesState) -> dict:
    return {"messages": [agent_llm.invoke([AGENT_PROMPT] + state["messages"])]}

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(rag_tools))

In [ ]:
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

In [ ]:
rag_agent = builder.compile()
show_graph(rag_agent)

In [ ]:
for question in ["Hi there!", "I bought the course 3 days ago and watched 10%. Can I get a refund, and how fast?"]:
    result = rag_agent.invoke({"messages": [("user", question)]})
    searched = any(m.type == "tool" for m in result["messages"])
    print("Q:", question)
    print("Searched the handbook?", searched)
    print("A:", result["messages"][-1].content)
    print()

## 5. Self-correcting RAG

Real users don't type perfect questions. They type things like "that weekend zoom thing, what time?" The search might come back with the wrong chunks, and then the answer is wrong.

A careful student, when the pages they turned to don't help, doesn't just guess. They think "let me look for a better keyword" and search again. We'll teach our graph to do the same:

1. **retrieve** chunks
2. **grade** them: does this context actually help answer the question? (structured output, yes or no)
3. if yes, **generate** the answer
4. if no, **rewrite** the question into a better search query and try again, at most twice

```
  START -> retrieve -> grade --relevant--> generate -> END
              ^           |
              |      not relevant
              |           v
              +------- rewrite
```

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

In [ ]:
class Relevance(BaseModel):
    relevant: Literal["yes", "no"] = Field(description="yes if the context helps answer the question")

In [ ]:
grader = llm.with_structured_output(Relevance)

In [ ]:
class SmartRAGState(TypedDict):
    question: str             # the current search query (may be rewritten)
    original_question: str    # what the user actually asked
    context: list[Document]
    relevant: str
    rewrites: int
    answer: str

In [ ]:
def smart_retrieve(state: SmartRAGState) -> dict:
    return {
        "context": retriever.invoke(state["question"]),
        # remember the user's original wording the first time round
        "original_question": state.get("original_question") or state["question"],
    }

In [ ]:
def grade(state: SmartRAGState) -> dict:
    context_text = "\n".join(d.page_content for d in state["context"])
    result = grader.invoke(f"Question: {state['original_question']}\n\nContext:\n{context_text}")
    print(f"Graded search for '{state['question']}': relevant = {result.relevant}")
    return {"relevant": result.relevant}

In [ ]:
def rewrite(state: SmartRAGState) -> dict:
    prompt = (
        "Rewrite this question as a clear search query for an online course handbook "
        "(topics: refunds, certificates, live classes, support). Reply with the query only.\n\n"
        f"{state['original_question']}"
    )
    better_query = llm.invoke(prompt).content
    print("Rewrote the query to:", better_query)
    return {"question": better_query, "rewrites": state.get("rewrites", 0) + 1}

In [ ]:
def smart_generate(state: SmartRAGState) -> dict:
    # Reuse the generate function from section 3, but answer the ORIGINAL question
    return generate({"question": state["original_question"], "context": state["context"]})

In [ ]:
def after_grading(state: SmartRAGState) -> Literal["generate", "rewrite"]:
    if state["relevant"] == "yes":
        return "generate"
    if state.get("rewrites", 0) >= 2:
        return "generate"         # tried enough, answer with what we have
    return "rewrite"

In [ ]:
builder = StateGraph(SmartRAGState)

In [ ]:
builder.add_node("retrieve", smart_retrieve)
builder.add_node("grade", grade)
builder.add_node("rewrite", rewrite)
builder.add_node("generate", smart_generate)

In [ ]:
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", after_grading)
builder.add_edge("rewrite", "retrieve")
builder.add_edge("generate", END)

In [ ]:
smart_rag = builder.compile()
show_graph(smart_rag)

In [ ]:
result = smart_rag.invoke({"question": "that weekend zoom thing, what time?", "rewrites": 0})
print()
print("Answer:", result["answer"])

## 6. Common mistakes

**Mistake 1: Chunks that are too big or too small.**
Too big, and each result is full of unrelated text, which confuses the model and costs more. Too small, and a chunk loses its meaning ("within 5 working days" on its own means nothing). Somewhere around a few hundred characters, with a little overlap, is a good starting point. Experiment with your own documents.

**Mistake 2: Not telling the model to stay inside the context.**
Without "use ONLY the context" and "say I don't know", the model mixes in its own knowledge, and you can't tell which parts came from your documents.

**Mistake 3: Forgetting the source metadata.**
If you don't store `metadata={"source": ...}` when creating documents, you can't show users where an answer came from, and they have no way to check it.

**Mistake 4: Rebuilding the vector store on every question.**
`InMemoryVectorStore.from_documents(...)` embeds every chunk, and embedding costs money and time. Build the store once, and reuse the retriever. To add documents later, use `vector_store.add_documents(...)`.

**Mistake 5: No limit on rewrites.**
If a question truly isn't covered by your documents, the grader will keep saying "no" forever. Cap the rewrites, just like every other loop.

## 7. Practice

**Exercise 1.** Add a new document, `fees.md`: "The LangGraph course costs Rs. 2,999. Students get 20% off with the code CHAI20." Add it to the existing vector store with `vector_store.add_documents(...)`, then ask the simple `rag_app` about the price.

**Exercise 2.** Change the simple RAG graph so the result also includes `sources`, a sorted list of the unique file names that were used.

In [ ]:
# Solution 1
fees_doc = Document(
    page_content="Fees. The LangGraph course costs Rs. 2,999. Students get 20% off with the code CHAI20.",
    metadata={"source": "fees.md"},
)
vector_store.add_documents([fees_doc])

In [ ]:
result = rag_app.invoke({"question": "How much does the LangGraph course cost, and is there a discount?"})
print(result["answer"])

In [ ]:
# Solution 2
class RAGWithSources(RAGState):
    sources: list[str]

In [ ]:
def generate_with_sources(state: RAGWithSources) -> dict:
    update = generate(state)
    update["sources"] = sorted({d.metadata["source"] for d in state["context"]})
    return update

In [ ]:
builder = StateGraph(RAGWithSources)

In [ ]:
builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate_with_sources)

In [ ]:
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

In [ ]:
rag_with_sources = builder.compile()

In [ ]:
result = rag_with_sources.invoke({"question": "How do I verify a certificate?"})
print(result["answer"])
print("Sources:", result["sources"])

## Wrapping up

RAG is an open-book exam for the model: find the relevant chunks, put them in the prompt, generate the answer. The pipeline is split, embed, store, retrieve. Embeddings let the search work on **meaning** rather than exact words.

We made it smarter in two steps. **Agentic RAG** turns the retriever into a tool so the agent decides when and how often to search. **Self-correcting RAG** grades what it found and rewrites the query when the results aren't useful, with a limit on retries.

And in every version: tell the model to stay inside the context, to say "I don't know", and to cite its sources.

Tomorrow we tackle a different kind of memory. Day 7's memory lasts for one conversation. Tomorrow's lasts **across** conversations, so your app can remember that a user is vegetarian or prefers Hindi, even weeks later.